
# B6739485 อารีย์รัตน์ มุ่งยอดกลาง
HW3: สรุปผลการค้นหา (Search Algorithms) บนแผนที่โรมาเนีย

โจทย์: หาเส้นทางจาก **Arad** ไปยัง **Vaslui** โดยใช้อัลกอริทึมค้นหา 11 แบบ ตามที่กำหนดในสรุปลัพธ์การคำนวณ:

1. Breadth First Tree Search
2. Depth First Tree Search
3. Breadth First Search (Graph Search)
4. Depth First Graph Search
5. Best First Graph Search
6. Uniform Cost Search
7. Depth Limited Search
8. Iterative Deepening Search
9. Greedy Best First Search
10. A\*-Search
11. Recursive Best First Search

กราฟและระยะทาง (คิดเลขต้นทุน = ค่าบนเส้นเชื่อมในรูป) นำมาจากภาพแผนที่โรมาเนียที่แนบมา

## 1. กำหนดกราฟของแผนที่ (จากภาพ) และฟังก์ชัน Heuristic

In [1]:
import math, heapq
from collections import deque

# เส้นเชื่อมและต้นทุน (undirected) ตามภาพแผนที่โรมาเนีย
edges = [
    ('Oradea','Zerind',71), ('Oradea','Sibiu',151),
    ('Zerind','Arad',75),
    ('Arad','Sibiu',140), ('Arad','Timisoara',118),
    ('Timisoara','Lugoj',111),
    ('Lugoj','Mehadia',70),
    ('Mehadia','Drobeta',75),
    ('Drobeta','Craiova',120),
    ('Craiova','RimnicuVilcea',146), ('Craiova','Pitesti',138),
    ('RimnicuVilcea','Sibiu',80), ('RimnicuVilcea','Pitesti',97),
    ('Sibiu','Fagaras',99),
    ('Fagaras','Bucharest',211),
    ('Pitesti','Bucharest',101),
    ('Bucharest','Giurgiu',90), ('Bucharest','Urziceni',85),
    ('Urziceni','Hirsova',98), ('Hirsova','Eforie',86),
    ('Urziceni','Vaslui',142),
    ('Vaslui','Iasi',92),
    ('Iasi','Neamt',87),
]

graph = {}
for a, b, c in edges:
    graph.setdefault(a, {})[b] = c
    graph.setdefault(b, {})[a] = c

# พิกัด (lat, lon) โดยประมาณของแต่ละเมือง ใช้คำนวณ heuristic แบบเส้นตรง (straight-line distance)
coords = {
    'Arad':(46.1866,21.3123), 'Zerind':(46.6225,21.5033), 'Oradea':(47.0722,21.9210),
    'Sibiu':(45.7983,24.1256), 'Fagaras':(45.8416,24.9731), 'Timisoara':(45.7489,21.2087),
    'Lugoj':(45.6900,21.9033), 'Mehadia':(44.9040,22.3644), 'Drobeta':(44.6250,22.6560),
    'Craiova':(44.3302,23.7949), 'RimnicuVilcea':(45.1047,24.3750), 'Pitesti':(44.8565,24.8692),
    'Bucharest':(44.4268,26.1025), 'Giurgiu':(43.9037,25.9699), 'Urziceni':(44.7167,26.6333),
    'Hirsova':(44.6833,27.9500), 'Eforie':(44.0650,28.6520), 'Vaslui':(46.6407,27.7276),
    'Iasi':(47.1585,27.6014), 'Neamt':(46.9759,26.3819),
}

def haversine_km(p1, p2):
    lat1, lon1 = p1; lat2, lon2 = p2
    R = 6371.0
    dlat = math.radians(lat2 - lat1); dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2)
    return R * 2 * math.asin(math.sqrt(a))

START, GOAL = 'Arad', 'Vaslui'

def h(state):
    """Heuristic: ระยะทางเส้นตรง (straight-line distance) จาก state ไปยังเป้าหมาย Vaslui"""
    return haversine_km(coords[state], coords[GOAL])

## 2. โครงสร้าง Node และฟังก์ชันช่วยแสดงผล

In [2]:
class Node:
    def __init__(self, state, parent=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.path_cost = path_cost
        self.depth = 0 if parent is None else parent.depth + 1

    def path(self):
        node, out = self, []
        while node:
            out.append(node.state)
            node = node.parent
        return list(reversed(out))

    def expand(self, graph):
        return [Node(nbr, self, self.path_cost + cost)
                for nbr, cost in graph.get(self.state, {}).items()]

    def __repr__(self):
        return f"<Node {self.state}>"

results = {}  # เก็บผลลัพธ์ของทุกอัลกอริทึมไว้เปรียบเทียบท้ายสุด

def report(name, result):
    node, expanded = result
    results[name] = (node, expanded)
    if node is None:
        print(f"{name}\n  -> ไม่พบเส้นทาง (expanded={expanded})")
    else:
        print(f"{name}\n  Path : {' -> '.join(node.path())}\n  Cost : {node.path_cost}\n  Expanded nodes: {expanded}")

## 3. Breadth First Tree Search

สืบค้นแบบ Tree Search (ไม่มี explored set) ด้วยคิว FIFO ขยายทีละ level จนกว่าจะเจอ goal

In [3]:
def breadth_first_tree_search(graph, start, goal):
    frontier = deque([Node(start)])
    expanded = 0
    while frontier:
        node = frontier.popleft()
        expanded += 1
        if node.state == goal:
            return node, expanded
        frontier.extend(node.expand(graph))
    return None, expanded

report("1. Breadth First Tree Search", breadth_first_tree_search(graph, START, GOAL))

1. Breadth First Tree Search
  Path : Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui
  Cost : 677
  Expanded nodes: 207


## 4. Depth First Tree Search

สืบค้นแบบ Tree Search ด้วยสแต็ก (LIFO) ไม่มี explored set จึงอาจวนซ้ำสเตตเดิมได้

In [4]:
def depth_first_tree_search(graph, start, goal):
    frontier = [Node(start)]
    expanded = 0
    while frontier:
        node = frontier.pop()
        expanded += 1
        if node.state == goal:
            return node, expanded
        frontier.extend(node.expand(graph))
    return None, expanded

report("2. Depth First Tree Search", depth_first_tree_search(graph, START, GOAL))

2. Depth First Tree Search
  Path : Arad -> Timisoara -> Lugoj -> Mehadia -> Drobeta -> Craiova -> Pitesti -> Bucharest -> Urziceni -> Vaslui
  Cost : 960
  Expanded nodes: 10


## 5. Breadth First Search (Graph Search)

เหมือน BFS Tree Search แต่เพิ่ม explored set กันไม่ให้ขยายสเตตที่เจอไปแล้ว

In [5]:
def breadth_first_graph_search(graph, start, goal):
    node = Node(start)
    if node.state == goal:
        return node, 0
    frontier = deque([node])
    explored = {start}
    expanded = 0
    while frontier:
        node = frontier.popleft()
        expanded += 1
        for child in node.expand(graph):
            if child.state not in explored:
                if child.state == goal:
                    return child, expanded
                explored.add(child.state)
                frontier.append(child)
    return None, expanded

report("3. Breadth First Search", breadth_first_graph_search(graph, START, GOAL))

3. Breadth First Search
  Path : Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui
  Cost : 677
  Expanded nodes: 15


## 6. Depth First Graph Search

DFS แบบ Graph Search: มี explored set กันการวนซ้ำ

In [6]:
def depth_first_graph_search(graph, start, goal):
    frontier = [Node(start)]
    explored = set()
    expanded = 0
    while frontier:
        node = frontier.pop()
        if node.state == goal:
            return node, expanded
        if node.state in explored:
            continue
        expanded += 1
        explored.add(node.state)
        frontier.extend(c for c in node.expand(graph) if c.state not in explored)
    return None, expanded

report("4. Depth First Graph Search", depth_first_graph_search(graph, START, GOAL))

4. Depth First Graph Search
  Path : Arad -> Timisoara -> Lugoj -> Mehadia -> Drobeta -> Craiova -> Pitesti -> Bucharest -> Urziceni -> Vaslui
  Cost : 960
  Expanded nodes: 9


## 7. Best First Graph Search (แบบทั่วไป)

อัลกอริทึมกลางที่ใช้ priority queue ตามฟังก์ชัน f(node) ใดๆ — ใช้เป็นฐานให้ UCS, Greedy, A* ด้านล่าง

In [7]:
def best_first_graph_search(graph, start, goal, f):
    counter = 0
    node = Node(start)
    frontier = [(f(node), counter, node)]
    frontier_best = {start: f(node)}
    explored = set()
    expanded = 0
    while frontier:
        _, _, node = heapq.heappop(frontier)
        if node.state == goal:
            return node, expanded
        expanded += 1
        explored.add(node.state)
        for child in node.expand(graph):
            fc = f(child)
            if child.state not in explored and child.state not in frontier_best:
                counter += 1
                heapq.heappush(frontier, (fc, counter, child))
                frontier_best[child.state] = fc
            elif child.state in frontier_best and fc < frontier_best[child.state]:
                counter += 1
                heapq.heappush(frontier, (fc, counter, child))
                frontier_best[child.state] = fc
    return None, expanded

# ตัวอย่าง: ใช้ h(state) ตรงๆ เป็น f (คล้าย Greedy) เพื่อสาธิต Best First Graph Search
report("5. Best First Graph Search", best_first_graph_search(graph, START, GOAL, lambda n: h(n.state)))

5. Best First Graph Search
  Path : Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui
  Cost : 677
  Expanded nodes: 5


## 8. Uniform Cost Search

Best First Graph Search โดยใช้ f(n) = g(n) (ต้นทุนสะสมจากจุดเริ่ม) — รับประกันเส้นทางที่ต้นทุนต่ำสุด

In [8]:
def uniform_cost_search(graph, start, goal):
    return best_first_graph_search(graph, start, goal, lambda n: n.path_cost)

report("6. Uniform Cost Search", uniform_cost_search(graph, START, GOAL))

6. Uniform Cost Search
  Path : Arad -> Sibiu -> RimnicuVilcea -> Pitesti -> Bucharest -> Urziceni -> Vaslui
  Cost : 645
  Expanded nodes: 17


## 9. Depth Limited Search

DFS ที่จำกัดความลึกสูงสุด (limit) ป้องกันการค้นลึกไม่มีที่สิ้นสุด

In [9]:
def depth_limited_search(graph, start, goal, limit=50):
    counter = [0]
    def rec(node, limit):
        counter[0] += 1
        if node.state == goal:
            return node
        if limit == 0:
            return 'cutoff'
        cutoff = False
        for child in node.expand(graph):
            r = rec(child, limit - 1)
            if r == 'cutoff':
                cutoff = True
            elif r is not None:
                return r
        return 'cutoff' if cutoff else None
    result = rec(Node(start), limit)
    return (result if result != 'cutoff' else None), counter[0]

report("7. Depth Limited Search (limit=10)", depth_limited_search(graph, START, GOAL, 10))

7. Depth Limited Search (limit=10)
  Path : Arad -> Zerind -> Oradea -> Zerind -> Oradea -> Sibiu -> RimnicuVilcea -> Pitesti -> Bucharest -> Urziceni -> Vaslui
  Cost : 944
  Expanded nodes: 453


## 10. Iterative Deepening Search

เรียก Depth Limited Search ซ้ำโดยเพิ่ม limit ขึ้นทีละ 1 จนพบคำตอบ

In [10]:
def iterative_deepening_search(graph, start, goal, max_depth=50):
    total = 0
    for depth in range(max_depth):
        result, expanded = depth_limited_search(graph, start, goal, depth)
        total += expanded
        if result is not None:
            return result, total
    return None, total

report("8. Iterative Deepening Search", iterative_deepening_search(graph, START, GOAL))

8. Iterative Deepening Search
  Path : Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui
  Cost : 677
  Expanded nodes: 326


## 11. Greedy Best First Search

Best First Graph Search โดยใช้ f(n) = h(n) เท่านั้น (ไม่สนต้นทุนที่เดินมาแล้ว)

In [11]:
def greedy_best_first_search(graph, start, goal, h):
    return best_first_graph_search(graph, start, goal, lambda n: h(n.state))

report("9. Greedy Best First Search", greedy_best_first_search(graph, START, GOAL, h))

9. Greedy Best First Search
  Path : Arad -> Sibiu -> Fagaras -> Bucharest -> Urziceni -> Vaslui
  Cost : 677
  Expanded nodes: 5


## 12. A\*-Search

Best First Graph Search โดยใช้ f(n) = g(n) + h(n) — เส้นทางที่ได้เป็นเส้นทางที่ต้นทุนต่ำสุด เพราะ heuristic (ระยะเส้นตรง) เป็น admissible

In [12]:
def astar_search(graph, start, goal, h):
    return best_first_graph_search(graph, start, goal, lambda n: n.path_cost + h(n.state))

report("10. A*-Search", astar_search(graph, START, GOAL, h))

10. A*-Search
  Path : Arad -> Sibiu -> RimnicuVilcea -> Pitesti -> Bucharest -> Urziceni -> Vaslui
  Cost : 645
  Expanded nodes: 12


## 13. Recursive Best First Search

รูปแบบเวอร์ชันประหยัดความจำของ A* ใช้ f-value และ backtracking limit ในการค้นแบบ recursive

In [13]:
def rbfs(graph, start, goal, h):
    counter = [0]
    def RBFS(node, f_limit):
        counter[0] += 1
        if node.state == goal:
            return node, 0
        successors = node.expand(graph)
        if not successors:
            return None, float('inf')
        for s in successors:
            s.f = max(s.path_cost + h(s.state), node.f)
        while True:
            successors.sort(key=lambda n: n.f)
            best = successors[0]
            if best.f > f_limit:
                return None, best.f
            alt = successors[1].f if len(successors) > 1 else float('inf')
            result, best.f = RBFS(best, min(f_limit, alt))
            if result is not None:
                return result, best.f
    root = Node(start)
    root.f = h(start)
    result, _ = RBFS(root, float('inf'))
    return result, counter[0]

report("11. Recursive Best First Search", rbfs(graph, START, GOAL, h))

11. Recursive Best First Search
  Path : Arad -> Sibiu -> RimnicuVilcea -> Pitesti -> Bucharest -> Urziceni -> Vaslui
  Cost : 645
  Expanded nodes: 144


## 14. สรุปเปรียบเทียบผลลัพธ์ทั้ง 11 อัลกอริทึม

In [14]:
print(f"{'Algorithm':38s} {'Path Cost':>10s} {'Nodes Expanded':>15s}")
print('-'*68)
for name, (node, expanded) in results.items():
    cost = node.path_cost if node else '-'
    print(f"{name:38s} {str(cost):>10s} {expanded:>15d}")

Algorithm                               Path Cost  Nodes Expanded
--------------------------------------------------------------------
1. Breadth First Tree Search                  677             207
2. Depth First Tree Search                    960              10
3. Breadth First Search                       677              15
4. Depth First Graph Search                   960               9
5. Best First Graph Search                    677               5
6. Uniform Cost Search                        645              17
7. Depth Limited Search (limit=10)            944             453
8. Iterative Deepening Search                 677             326
9. Greedy Best First Search                   677               5
10. A*-Search                                 645              12
11. Recursive Best First Search               645             144


### สรุปผลแยกตามแต่ละอัลกอริทึม (Arad → Vaslui)

**1. Breadth First Tree Search**
- เส้นทาง: Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
- ต้นทุน: 677 | ขยายโหนด: 207
- ไม่มี explored set จึงขยายซ้ำสเตตเดิมได้เรื่อยๆ ทำให้ขยายโหนดมากที่สุดในกลุ่ม BFS แต่เพราะเป็น BFS จึงยังได้ต้นทุนต่ำสุด**ในแบบไม่ถ่วงน้ำหนัก** (จำนวนเมืองน้อยสุด)

**2. Depth First Tree Search**
- เส้นทาง: Arad → Timisoara → Lugoj → Mehadia → Drobeta → Craiova → Pitesti → Bucharest → Urziceni → Vaslui
- ต้นทุน: 960 | ขยายโหนด: 10
- ขยายโหนดน้อยที่สุดในทุกอัลกอริทึม เพราะดิ่งลงไปทางเดียวจนเจอ goal โดยบังเอิญไม่ต้อง backtrack แต่ต้นทุนสูงที่สุด เพราะไม่สนใจต้นทุนหรือ heuristic เลย

**3. Breadth First Search (Graph Search)**
- เส้นทาง: Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
- ต้นทุน: 677 | ขยายโหนด: 15
- ผลเหมือนข้อ 1 แต่ขยายโหนดน้อยกว่ามาก (207 → 15) เพราะมี explored set ตัดสเตตที่เจอไปแล้วออก ทำให้ไม่เสียเวลาขยายซ้ำ

**4. Depth First Graph Search**
- เส้นทาง: Arad → Timisoara → Lugoj → Mehadia → Drobeta → Craiova → Pitesti → Bucharest → Urziceni → Vaslui
- ต้นทุน: 960 | ขยายโหนด: 9
- ผลเหมือนข้อ 2 แต่ขยายโหนดน้อยลงอีกเล็กน้อย (10 → 9) เพราะ explored set ช่วยตัดกิ่งที่ซ้ำออกไปได้บ้าง

**5. Best First Graph Search (f = h(n))**
- เส้นทาง: Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
- ต้นทุน: 677 | ขยายโหนด: 5
- ขยายโหนดน้อยมาก เพราะเลือกโหนดที่ heuristic บอกว่าใกล้ Vaslui ที่สุดก่อนเสมอ ในเคสนี้พาไปเจอ goal ตรงๆ แต่บังเอิญได้ต้นทุนเท่ากับ BFS (677) ไม่ใช่ต้นทุนต่ำสุดจริง (645)

**6. Uniform Cost Search**
- เส้นทาง: Arad → Sibiu → RimnicuVilcea → Pitesti → Bucharest → Urziceni → Vaslui
- ต้นทุน: 645 | ขยายโหนด: 17
- เป็นอัลกอริทึมแรกที่เจอ**ต้นทุนต่ำสุดจริง (645)** เพราะเลือกขยายตามต้นทุนสะสม (g) แต่ต้องขยายโหนดมากกว่ากลุ่ม heuristic เพราะไม่มีตัวช่วยมองไปข้างหน้าเลย

**7. Depth Limited Search (limit = 10)**
- เส้นทาง: Arad → Zerind → Oradea → Zerind → Oradea → Sibiu → RimnicuVilcea → Pitesti → Bucharest → Urziceni → Vaslui
- ต้นทุน: 944 | ขยายโหนด: 453
- ขยายโหนดมากที่สุดในทุกอัลกอริทึม เพราะเป็น Tree Search ที่ไม่มี explored set จึงวนเข้า-ออก Zerind/Oradea ซ้ำได้ในขอบเขต limit และเส้นทางที่ได้ยังมีสเตตซ้ำ (ไม่ใช่เส้นทางที่ดีที่สุด)

**8. Iterative Deepening Search**
- เส้นทาง: Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
- ต้นทุน: 677 | ขยายโหนด: 326 (รวมทุกรอบ limit=0..4)
- ได้ผลเหมือน BFS (677) เพราะเป็น BFS ที่ทำในรูปแบบ depth-limited ซ้ำหลายรอบ จึงเสียโหนดที่ขยายไปมากในรอบ limit ต่ำๆ ที่ยังหาไม่พบคำตอบ

**9. Greedy Best First Search**
- เส้นทาง: Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
- ต้นทุน: 677 | ขยายโหนด: 5
- ผลเหมือนข้อ 5 (เพราะใช้ f = h(n) เหมือนกัน) ขยายโหนดน้อยที่สุดในกลุ่มที่หาคำตอบสำเร็จ แต่ไม่รับประกันต้นทุนต่ำสุด

**10. A\*-Search**
- เส้นทาง: Arad → Sibiu → RimnicuVilcea → Pitesti → Bucharest → Urziceni → Vaslui
- ต้นทุน: 645 | ขยายโหนด: 12
- ได้**ต้นทุนต่ำสุดเท่ากับ UCS (645)** แต่ขยายโหนดน้อยกว่า UCS มาก (17 → 12) เพราะใช้ heuristic ช่วยตัดกิ่งที่ไม่น่าไปถึง goad ทิ้งไปก่อน ถือว่าเป็นตัวที่คุ้มค่าที่สุดในกลุ่ม optimal

**11. Recursive Best First Search**
- เส้นทาง: Arad → Sibiu → RimnicuVilcea → Pitesti → Bucharest → Urziceni → Vaslui
- ต้นทุน: 645 | ขยายโหนด: 144
- ได้ต้นทุนต่ำสุดเท่ากับ A\*/UCS (645) เช่นกัน แต่ขยายโหนดมากกว่า A\* มาก (12 → 144) เพราะต้อง backtrack กลับไปคำนวณซ้ำหลายรอบเพื่อประหยัดหน่วยความจำ (แลกเวลากับพื้นที่หน่วยความจำ)